In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
STEP4_2_4_4_generate_missing_data.py
----------------------------------
目的: 論文執筆に必要な「詳細データ」と「サマリーデータ」を確実に生成する。
      特に 'linear_top3_k50' の詳細解析を強制実行する。
"""

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

# ========= 設定 =========
RUN_ID = "20251130_153500"
ROOT = Path("/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127")

CLUSTERING_BASE = ROOT / "sub" / "03_clustering_STEP3.2_signlessCorr" / f"run_{RUN_ID}"
SUMMARY_BASE    = ROOT / "sub" / "04_summary_STEP3.2_signlessCorr" / f"run_{RUN_ID}"

# 入力ファイル
FEATURES_OH = ROOT / "data" / "raw" / "features_OH_onlyMat.csv"
FEATURES_FP = ROOT / "data" / "raw" / "features_FP_onlyMat.csv"

# ラベルファイル (変数用ではなくサンプル用を見る)
# ※今回は「Cluster 19」の中身を知りたいので、SAMPLESの解析が必要
FIGS_OH_DIR = CLUSTERING_BASE / "samples" / "A_OH_plus_others" / "labels"
FIGS_FP_DIR = CLUSTERING_BASE / "samples" / "B_FP_plus_others" / "labels"

# 出力先
OUT_ROOT = ROOT / "Thesis_Figures" / f"run_{RUN_ID}" / "paper_4.2.4.4_OHFP"
OUT_CSV = OUT_ROOT / "reverse" / "analysis_csv"
OUT_BIDIR = OUT_ROOT / "bidirectional_summary"

for d in [OUT_CSV, OUT_BIDIR]:
    d.mkdir(parents=True, exist_ok=True)

# 解析対象の条件 (強制実行)
TARGET_CONDITIONS = [
    ("linear", "top3", "k50"),  # これが本命
    ("linear", "top3", "k25"),
    ("linear", "top3", "k10")
]

print(f"Target Run ID: {RUN_ID}")

# ========= Helper =========
def read_csv_quiet(path):
    if not path.exists(): return None
    try: return pd.read_csv(path)
    except: return pd.read_csv(path, encoding="cp932")

def load_features_bin(path):
    df = read_csv_quiet(path)
    if df is None: return None
    if df.columns[0].lower()!="sample_id":
        df = df.rename(columns={df.columns[0]:"sample_id"})
    X = df.set_index("sample_id").apply(pd.to_numeric, errors="coerce").fillna(0.0)
    return (X!=0).astype(int)

FP_ROLE_MAP = {None:"inM", "1":"ip1M", "2":"ip2M", "3":"nM", "4":"p1M", "5":"p2M"}
FP_PAT = re.compile(r"^X(?P<num>\d+)(?:\.(?P<suf>[1-5]))?$", flags=re.IGNORECASE)

def fp_var_display_name(fp_var):
    s = str(fp_var).strip()
    m = FP_PAT.match(s)
    if not m: return fp_var
    num = m.group("num")
    suf = m.group("suf")
    role = FP_ROLE_MAP.get(suf, "inM")
    return f"X{num}[{role}]"

# ========= Main Logic =========
def main():
    print("Loading Features...")
    if not FEATURES_OH.exists() or not FEATURES_FP.exists():
        print("[ERROR] Features not found. Cannot analyze content.")
        return

    Xoh = load_features_bin(FEATURES_OH)
    Xfp = load_features_bin(FEATURES_FP)
    
    # 共通サンプル (ID合わせ)
    Xoh.index = Xoh.index.astype(str)
    Xfp.index = Xfp.index.astype(str)
    common = Xoh.index.intersection(Xfp.index)
    Xoh = Xoh.loc[common]; Xfp = Xfp.loc[common]
    print(f"Common samples: {len(common)}")

    # 条件ループ
    for method, dim, k_str in TARGET_CONDITIONS:
        cond_name = f"{method}_{dim}_{k_str}"
        print(f"\nProcessing: {cond_name} ...")
        
        # ラベルファイルを探す
        # ClusterAssign_linear_top3_k50_samples_A_OH_plus_others_20251130_153500.csv
        fname_oh = f"ClusterAssign_{method}_{dim}_{k_str}_samples_A_OH_plus_others_{RUN_ID}.csv"
        fname_fp = f"ClusterAssign_{method}_{dim}_{k_str}_samples_B_FP_plus_others_{RUN_ID}.csv"
        
        path_oh = FIGS_OH_DIR / fname_oh
        path_fp = FIGS_FP_DIR / fname_fp
        
        lab_oh_df = read_csv_quiet(path_oh)
        lab_fp_df = read_csv_quiet(path_fp)
        
        if lab_oh_df is None or lab_fp_df is None:
            print(f"  [WARN] Label file not found: {fname_oh} or {fname_fp}")
            continue
            
        # IDとCluster列を取得してSeries化
        def to_series(df):
            vcol = "ID" if "ID" in df.columns else df.columns[0]
            ccol = "Cluster" if "Cluster" in df.columns else df.columns[-1]
            return pd.Series(df[ccol].values, index=df[vcol].astype(str).values)

        s_oh = to_series(lab_oh_df)
        s_fp = to_series(lab_fp_df)
        
        # 共通IDでフィルタ
        valid_ids = common.intersection(s_oh.index).intersection(s_fp.index)
        if len(valid_ids) == 0: continue
        
        # --- 1. OH Cluster の中身解析 (Cluster 19 はどんな材料か？) ---
        # 各クラスターに含まれるサンプル群において、どの材料(OH変数)が支配的か？
        
        oh_cluster_details = []
        for cid in np.unique(s_oh.loc[valid_ids]):
            # このクラスターのサンプルID
            ids_in_cluster = s_oh[s_oh == cid].index.intersection(valid_ids)
            if len(ids_in_cluster) == 0: continue
            
            # そのサンプル群での材料使用率 (平均ベクトル)
            vec = Xoh.loc[ids_in_cluster].mean(axis=0)
            # 使用率が高い順にトップ5を取得
            top_mats = vec[vec > 0].sort_values(ascending=False).head(5)
            
            mat_info = "; ".join([f"{mat}({prob:.2f})" for mat, prob in top_mats.items()])
            major_mat = top_mats.index[0] if len(top_mats) > 0 else "None"
            
            oh_cluster_details.append({
                "Cluster_ID": cid,
                "n_samples": len(ids_in_cluster),
                "Major_Material": major_mat,
                "Top5_Materials_Prob": mat_info
            })
            
        df_oh_detail = pd.DataFrame(oh_cluster_details)
        out_name1 = f"Detail_OH_Clusters_{cond_name}.csv"
        df_oh_detail.to_csv(OUT_CSV / out_name1, index=False)
        print(f"  [SAVE] {out_name1}")

        # --- 2. FP Cluster の中身解析 (Cluster 18 はどんなフラグメントか？) ---
        # 各クラスターに含まれるサンプル群において、どのフラグメント(FP変数)が支配的か？
        
        fp_cluster_details = []
        fp_vars = [c for c in Xfp.columns if c.startswith("X")] # FP変数のみ
        
        for cid in np.unique(s_fp.loc[valid_ids]):
            ids_in_cluster = s_fp[s_fp == cid].index.intersection(valid_ids)
            if len(ids_in_cluster) == 0: continue
            
            vec = Xfp.loc[ids_in_cluster, fp_vars].mean(axis=0)
            top_frags = vec[vec > 0].sort_values(ascending=False).head(5)
            
            frag_info = "; ".join([f"{fp_var_display_name(f)}({p:.2f})" for f, p in top_frags.items()])
            major_frag = fp_var_display_name(top_frags.index[0]) if len(top_frags) > 0 else "None"
            
            fp_cluster_details.append({
                "Cluster_ID": cid,
                "n_samples": len(ids_in_cluster),
                "Major_Fragment": major_frag,
                "Top5_Fragments_Prob": frag_info
            })
            
        df_fp_detail = pd.DataFrame(fp_cluster_details)
        out_name2 = f"Detail_FP_Clusters_{cond_name}.csv"
        df_fp_detail.to_csv(OUT_CSV / out_name2, index=False)
        print(f"  [SAVE] {out_name2}")

    print("\n✅ Data Generation Complete.")

if __name__ == "__main__":
    main()

Target Run ID: 20251130_153500
Loading Features...
Common samples: 1046

Processing: linear_top3_k50 ...
  [SAVE] Detail_OH_Clusters_linear_top3_k50.csv
  [SAVE] Detail_FP_Clusters_linear_top3_k50.csv

Processing: linear_top3_k25 ...
  [SAVE] Detail_OH_Clusters_linear_top3_k25.csv
  [SAVE] Detail_FP_Clusters_linear_top3_k25.csv

Processing: linear_top3_k10 ...
  [SAVE] Detail_OH_Clusters_linear_top3_k10.csv
  [SAVE] Detail_FP_Clusters_linear_top3_k10.csv

✅ Data Generation Complete.


In [2]:
import pandas as pd
import os
import glob

# ==========================================
# 1. ファイル設定
# ==========================================
def find_file(pattern):
    files = glob.glob(pattern)
    return files[0] if files else None

file_thesis_A = find_file('*ThesisData_A_OH_plus_others_MDS_Targets*.csv')
file_thesis_B = find_file('*ThesisData_B_FP_plus_others_MDS_Targets*.csv')
file_labels_A = find_file('*cluster_labels_matrix_samples_A_OH_plus_others*.csv')
file_labels_B = find_file('*cluster_labels_matrix_samples_B_FP_plus_others*.csv')

# Detail files
file_detail_OH_k50 = 'Detail_OH_Clusters_linear_top3_k50.csv'
file_detail_FP_k50 = 'Detail_FP_Clusters_linear_top3_k50.csv'

# Output filenames
OUT_OH_FULL = 'Table_Appendix_OH_Full_k50.csv'
OUT_FP_FULL = 'Table_Appendix_FP_Full_k50.csv'

# ==========================================
# 2. データ読み込み & 結合
# ==========================================
def load_and_merge(f_thesis, f_labels):
    if not f_thesis or not f_labels: return None
    df_t = pd.read_csv(f_thesis)
    if 'Unnamed: 0' in df_t.columns:
        df_t.rename(columns={'Unnamed: 0': 'ID'}, inplace=True)
    df_l = pd.read_csv(f_labels)
    return pd.merge(df_t, df_l, on='ID', how='inner')

def load_detail(path):
    return pd.read_csv(path) if os.path.exists(path) else None

df_A = load_and_merge(file_thesis_A, file_labels_A)
df_B = load_and_merge(file_thesis_B, file_labels_B)
det_OH = load_detail(file_detail_OH_k50)
det_FP = load_detail(file_detail_FP_k50)

# ==========================================
# 3. 集計関数 (全指標対応)
# ==========================================
def generate_full_table(df, col_k, detail_df, type_name):
    if df is None: return None
    
    # クラスターごとに集計
    targets = ['PCEmax', 'Jsc', 'Voc', 'FF']
    
    # 平均と標準偏差、件数を計算
    stats = df.groupby(col_k)[targets].agg(['mean', 'std', 'count'])
    
    rows = []
    for cid in stats.index:
        # 基本統計量
        n = stats.loc[cid, ('PCEmax', 'count')]
        pce_m = stats.loc[cid, ('PCEmax', 'mean')]
        pce_s = stats.loc[cid, ('PCEmax', 'std')]
        jsc_m = stats.loc[cid, ('Jsc', 'mean')]
        voc_m = stats.loc[cid, ('Voc', 'mean')]
        ff_m  = stats.loc[cid, ('FF', 'mean')]
        
        # 主要成分 (Detailから取得)
        desc = "N/A"
        if detail_df is not None:
            row = detail_df[detail_df['Cluster_ID'] == cid]
            if not row.empty:
                col_comp = 'Major_Material' if type_name == 'OH' else 'Major_Fragment'
                raw = str(row[col_comp].values[0])
                desc = raw.replace("SMILESsname", "").replace("p1M_", "").replace("nM_", "").replace("inM_", "")
        
        rows.append({
            "Cluster ID": cid,
            "N": int(n),
            "PCE (mean)": round(pce_m, 2),
            "PCE (std)": round(pce_s, 2),
            "Jsc": round(jsc_m, 2),
            "Voc": round(voc_m, 2),
            "FF": round(ff_m, 2),
            "Major Component": desc
        })
        
    # DataFrame化 & ソート
    res = pd.DataFrame(rows)
    res = res.sort_values("PCE (mean)", ascending=False) # PCE順
    return res

# ==========================================
# 4. 実行 & 保存
# ==========================================

# --- OH (Material) Full Table ---
if df_A is not None:
    print("\n--- OH (Material) Full Ranking (k=50) ---")
    tbl_oh = generate_full_table(df_A, 'linear_top3_k50', det_OH, 'OH')
    tbl_oh.to_csv(OUT_OH_FULL, index=False)
    print(tbl_oh.head(10)) # Top 10を表示
    print(f"Saved: {OUT_OH_FULL}")

# --- FP (Structure) Full Table ---
if df_B is not None:
    print("\n--- FP (Structure) Full Ranking (k=50) ---")
    tbl_fp = generate_full_table(df_B, 'linear_top3_k50', det_FP, 'FP')
    tbl_fp.to_csv(OUT_FP_FULL, index=False)
    print(tbl_fp.head(10)) # Top 10を表示
    print(f"Saved: {OUT_FP_FULL}")